In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
sys.path.append('../src/')
import regression_models as rm

In [2]:
# training dataset
raw_data = '../data/processed/2020_joined_norm_jeong.csv'
raw_df = pd.read_csv(raw_data)
# fill nan as 0
raw_df = raw_df.fillna(0)
training_data = raw_df.iloc[:, 6:]
target_data = raw_df.loc[:, 'candidatevotes_dem'] / raw_df.loc[:, 'totalvotes']

In [3]:
# 70/30 TTS and fixed random_state for dev
tts = train_test_split(training_data.values,
                       target_data.values,
                       test_size=0.3,
                       random_state=1776)
X_train, X_test, y_train, y_test = tts
feature_size = X_train.shape[-1]

In [4]:
model = rm.simple_regressr(feature_size)

/Users/yung/.local/share/mamba/envs/tf_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-30 23:22:51.495648: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-05-30 23:22:51.495679: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-30 23:22:51.495686: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-05-30 23:22:51.495703: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-30 23:22:51.495714: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory

In [5]:
hist = rm.compile_train_checkpoint(model, *tts, 500, '../models/test_linear.h5')

2026-05-30 23:22:51.918850: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
/Users/yung/.local/share/mamba/envs/tf_env/lib/python3.10/site-packages/keras/src/callbacks/model_checkpoint.py:276: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):


KeyboardInterrupt: 

In [7]:
label_data = '../data/Raw/countypres_2000-2024.csv'
label_data_df = pd.read_csv(label_data)

In [9]:
label_data_df.head()


,state,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,OTHER,OTHER,293.0,28281,20260225,TOTAL
1,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,65.0,28281,20260225,TOTAL
2,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,KAMALA D HARRIS,DEMOCRAT,7439.0,28281,20260225,TOTAL
3,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,20484.0,28281,20260225,TOTAL
4,ALABAMA,BALDWIN,2024,AL,1003.0,US PRESIDENT,OTHER,OTHER,1276.0,122249,20260225,TOTAL


In [ ]:
# training dataset
raw_data = '../data/processed/2020_joined_norm_jeong.csv'
raw_df = pd.read_csv(raw_data)
# fill nan as 0
raw_df = raw_df.fillna(0)
training_data = raw_df.iloc[:, 6:]
target_data = raw_df.loc[:, 'candidatevotes_dem'] / raw_df.loc[:, 'totalvotes']

In [3]:
# preview data
print('Training Dataset: ', training_data.shape)

Training Dataset:  (2305, 146)


In [4]:
# 70/30 TTS and fixed random_state for dev
tts = train_test_split(training_data.values,
                       target_data.values,
                       test_size=0.3,
                       random_state=1776)
X_train, X_test, y_train, y_test = tts

In [5]:
X_train.shape

(1613, 146)

In [7]:
feature_size = X_train.shape[-1]
# single hidden layer
regression_model = tf.keras.Sequential([
    layers.Dense(feature_size, input_shape=(feature_size,)),
    # softplus activation to keep positive
    layers.Dense(feature_size, activation='softplus'),
    # regreesion layear, no activation
    layers.Dense(1)
])

regression_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='mse'
)


In [8]:
# callback for saving weights with validation
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='../models/best_nn_linear_regression.h5',
    monitor='val_accuracy', # use validation as monitoring accuracy
    save_best_only=True,
    mode='max'              
)

training_history = regression_model.fit(
    X_train,
    y_train,
    epochs = 2000,
    verbose = 0,
    validation_data=(X_test, y_test),
    callbacks=[checkpoint_callback]
)

2026-05-27 11:48:01.147706: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
/Users/yung/.local/share/mamba/envs/tf_env/lib/python3.10/site-packages/keras/src/callbacks/model_checkpoint.py:276: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):


In [9]:
# load best model for metric
regression_model.load_weights('../models/best_nn_linear_regression.h5',)
pred = regression_model.predict(X_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [10]:
R2_metric = keras.metrics.R2Score()
R2_metric.update_state(y_test, pred)
R2_score = R2_metric.result()
MSE_metric = tf.keras.metrics.MeanSquaredError()
MSE_metric.update_state(y_test, pred)
mse_score = MSE_metric.result()
# basic linear regression achieved R2 0.3679282597984309
# SVM Regression achieved R2 0.7614698877080794
print('R2 Score: ', R2_score)
print('MSE Score: ', mse_score)

R2 Score:  tf.Tensor(0.7284051, shape=(), dtype=float32)
MSE Score:  tf.Tensor(0.006850635, shape=(), dtype=float32)


In [9]:
training_history.history

{'loss': [0.138778954744339,
  0.13368673622608185,
  0.13335378468036652,
  0.13327744603157043,
  0.1333799660205841,
  0.1332789808511734,
  0.13324372470378876,
  0.13325965404510498,
  0.13329391181468964,
  0.1333622932434082,
  0.1333107352256775,
  0.1332564800977707,
  2.157991409301758,
  0.1357322484254837,
  0.1369675099849701,
  0.1369674801826477,
  0.1369674950838089,
  0.1369674950838089,
  0.1369674950838089,
  0.1369675099849701,
  0.1369674950838089,
  0.1369674950838089,
  0.1369675099849701,
  0.1369675099849701,
  0.1369675099849701,
  0.1369674950838089,
  0.1369674950838089,
  0.1369675099849701,
  0.1369674801826477,
  0.1369674950838089,
  0.1369675099849701,
  0.1369674950838089,
  0.1369675099849701,
  0.1369675099849701,
  0.1369674950838089,
  0.1369674950838089,
  0.1369674950838089,
  0.1369674801826477,
  0.1369675099849701,
  0.1369675248861313,
  0.1369674950838089,
  0.1369674950838089,
  0.1369675099849701,
  0.1369674801826477,
  0.1369674801826477

In [14]:
# two hidden layer with compressing embedding space by half
comp_reg_model = tf.keras.Sequential([
    layers.Dense(feature_size, input_shape=(feature_size,)),
    # softplus activation to keep positive
    layers.Dense(int(feature_size/2), activation='softplus'),
    layers.Dense(int(feature_size/2), activation='softplus'),
    # regreesion layear, no activation
    layers.Dense(1)
])

comp_reg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='mse'
)

# callback for saving weights with validation
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='../models/best_nn_comp_linear_regression.h5',
    monitor='val_accuracy', # use validation as monitoring accuracy
    save_best_only=True,
    mode='max'              
)

comp_training_history = comp_reg_model.fit(
    X_train,
    y_train,
    epochs = 2000,
    verbose = 0,
    validation_data=(X_test, y_test),
    callbacks=[checkpoint_callback]
)

/Users/yung/.local/share/mamba/envs/tf_env/lib/python3.10/site-packages/keras/src/callbacks/model_checkpoint.py:276: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):


In [17]:
# load best model for metric
comp_reg_model.load_weights('../models/best_nn_comp_linear_regression.h5',)
comp_pred = comp_reg_model.predict(X_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [18]:
R2_metric = keras.metrics.R2Score()
R2_metric.update_state(y_test, comp_pred)
R2_score = R2_metric.result()
MSE_metric = tf.keras.metrics.MeanSquaredError()
MSE_metric.update_state(y_test, comp_pred)
mse_score = MSE_metric.result()
# basic linear regression achieved R2 0.3679282597984309
# SVM Regression achieved R2 0.7614698877080794
print('R2 Score: ', R2_score)
print('MSE Score: ', mse_score)

R2 Score:  tf.Tensor(0.7409699, shape=(), dtype=float32)
MSE Score:  tf.Tensor(0.006533704, shape=(), dtype=float32)


In [19]:
# more layers!
more_reg_model = tf.keras.Sequential([
    layers.Dense(feature_size, input_shape=(feature_size,)),
    # softplus activation to keep positive
    # compress and expand hidden space
    layers.Dense(feature_size, activation='softplus'),
    layers.Dense(int(feature_size/2), activation='softplus'),
    layers.Dense(feature_size, activation='softplus'),
    # regreesion layear, no activation
    layers.Dense(1)
])

more_reg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='mse'
)

# callback for saving weights with validation
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='../models/best_nn_more_linear_regression.h5',
    monitor='val_accuracy', # use validation as monitoring accuracy
    save_best_only=True,
    mode='max'              
)

more_training_history = more_reg_model.fit(
    X_train,
    y_train,
    epochs = 2000,
    verbose = 0,
    validation_data=(X_test, y_test),
    callbacks=[checkpoint_callback]
)

/Users/yung/.local/share/mamba/envs/tf_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/yung/.local/share/mamba/envs/tf_env/lib/python3.10/site-packages/keras/src/callbacks/model_checkpoint.py:276: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):


In [20]:
# load best model for metric
more_reg_model.load_weights('../models/best_nn_more_linear_regression.h5',)
more_pred = more_reg_model.predict(X_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [21]:
R2_metric = keras.metrics.R2Score()
R2_metric.update_state(y_test, more_pred)
R2_score = R2_metric.result()
MSE_metric = tf.keras.metrics.MeanSquaredError()
MSE_metric.update_state(y_test, more_pred)
mse_score = MSE_metric.result()
# basic linear regression achieved R2 0.3679282597984309
# SVM Regression achieved R2 0.7614698877080794
print('R2 Score: ', R2_score)
print('MSE Score: ', mse_score)

R2 Score:  tf.Tensor(0.7544497, shape=(), dtype=float32)
MSE Score:  tf.Tensor(0.006193694, shape=(), dtype=float32)


In [25]:
# high count of hidden state variables
hi_reg_model = tf.keras.Sequential([
    layers.Dense(feature_size, input_shape=(feature_size,)),
    # compress and expand hidden space
    # relue activation on hidden space for more discrete activation
    layers.Dense(feature_size*2, activation='softplus'),
    layers.Dense(feature_size, activation='softplus'),
    layers.Dense(int(feature_size/2), activation='softplus'),
    # softplus activation to keep positive
    layers.Dense(feature_size, activation='softplus'),
    # regreesion layear, no activation
    layers.Dense(1)
])

hi_reg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='mse'
)

# callback for saving weights with validation
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='../models/best_nn_hi_linear_regression.h5',
    monitor='val_accuracy', # use validation as monitoring accuracy
    save_best_only=True,
    mode='max'              
)

hi_training_history = hi_reg_model.fit(
    X_train,
    y_train,
    epochs = 2000,
    verbose = 0,
    validation_data=(X_test, y_test),
    callbacks=[checkpoint_callback]
)

In [26]:
# load best model for metric
hi_reg_model.load_weights('../models/best_nn_hi_linear_regression.h5',)
hi_pred = hi_reg_model.predict(X_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [27]:
R2_metric = keras.metrics.R2Score()
R2_metric.update_state(y_test, hi_pred)
R2_score = R2_metric.result()
MSE_metric = tf.keras.metrics.MeanSquaredError()
MSE_metric.update_state(y_test, hi_pred)
mse_score = MSE_metric.result()
# basic linear regression achieved R2 0.3679282597984309
# SVM Regression achieved R2 0.7614698877080794
print('R2 Score: ', R2_score)
print('MSE Score: ', mse_score)

R2 Score:  tf.Tensor(0.75437826, shape=(), dtype=float32)
MSE Score:  tf.Tensor(0.0061954954, shape=(), dtype=float32)
